# Guwahati LSTM - Stage 4 ML-Ready Data

This notebook trains:
- `CLOUD_Vertical_Visibility`
- `VIS_Visibility_MOR`

It uses the output of `scripts/10_prepare_guwahati_vertical_visibility_mlready.py` and a chronological split.

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "10_prepare_guwahati_vertical_visibility_mlready.py").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory")
DATA_FILE = PROJECT_ROOT / "07_lstm_ready" / "vertical_visibility_experiment" / "guwahati_mlready.csv"
ARTIFACT_DIR = PROJECT_ROOT / "10_models" / "guwahati_30_minutes"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = ["CLOUD_Vertical_Visibility", "VIS_Visibility_MOR"]
LOOKBACK = 240
HORIZON = 120
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
BATCH_SIZE = 256
EPOCHS = 50
PATIENCE = 8

FEATURE_COLUMNS = [
    "CLOUD_Cloud_Base_1", "CLOUD_Cloud_Base_2", "CLOUD_Cloud_Base_3",
    "CLOUD_OCTA1", "CLOUD_OCTA2", "CLOUD_OCTA3", "CLOUD_OCTA4",
    "PTU_Pressure_QNH", "PTU_Temperature", "PTU_DewPoint", "PTU_Humidity",
    "PW_Rain_Intensity", "PW_Rain_Sum", "VIS_RVR",
    "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos", "WIND_Wind_Speed", "WIND_Wind_Gust",
    "Hour_Sin", "Hour_Cos", "CLOUD_SENSOR_ON", "PTU_SENSOR_ON", "PW_SENSOR_ON",
    "VIS_SENSOR_ON", "WIND_SENSOR_ON", "CLOUD_Cloud_Base_1_IMPUTED",
    "CLOUD_Cloud_Base_2_IMPUTED", "CLOUD_Cloud_Base_3_IMPUTED",
    "CLOUD_Vertical_Visibility_IMPUTED", "CLOUD_LONG_GAP", "CLOUD_VERTICAL_LONG_GAP", "ROW_GAP",
]

assert DATA_FILE.exists(), DATA_FILE
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Data:", DATA_FILE)


PyTorch: 2.5.1
Device: cuda
Data: <repository>/07_lstm_ready/vertical_visibility_experiment/guwahati_mlready.csv


In [2]:
df = pd.read_csv(DATA_FILE, parse_dates=["Datetime"], dayfirst=True)
df = df.sort_values("Datetime").drop_duplicates("Datetime")

required_raw = sorted(set(FEATURE_COLUMNS + TARGET_COLUMNS) - {
    "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos", "ROW_GAP"
})
required_raw = sorted(set(required_raw) | {"WIND_Wind_Dir"})
missing = sorted(set(required_raw) - set(df.columns))
assert not missing, f"Missing raw columns: {missing}"

numeric_columns = sorted((set(FEATURE_COLUMNS + TARGET_COLUMNS) - {
    "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos", "ROW_GAP"
}) & set(df.columns))
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
df.loc[df["WIND_Wind_Speed"] < 0, "WIND_Wind_Speed"] = np.nan
df.loc[df["WIND_Wind_Gust"] < 0, "WIND_Wind_Gust"] = np.nan
df["WIND_Wind_Dir_Sin"] = np.sin(np.deg2rad(pd.to_numeric(df["WIND_Wind_Dir"], errors="coerce")))
df["WIND_Wind_Dir_Cos"] = np.cos(np.deg2rad(pd.to_numeric(df["WIND_Wind_Dir"], errors="coerce")))
df = df.set_index("Datetime")
df["ROW_GAP"] = df.index.to_series().diff().dt.total_seconds().gt(15).fillna(False)

print("Rows:", len(df), "range:", df.index.min(), df.index.max())
print("Valid cloud labels:", int(df["CLOUD_Vertical_Visibility"].notna().sum()))
print("Valid MOR labels:", int(df["VIS_Visibility_MOR"].notna().sum()))


Rows: 2848084 range: 2024-08-11 00:00:00 2026-01-31 10:08:45
Valid cloud labels: 2848084
Valid MOR labels: 1889019


In [3]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

train_end = int(len(df) * TRAIN_RATIO)
validation_end = int(len(df) * (TRAIN_RATIO + VALIDATION_RATIO))
feature_frame = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
feature_medians = feature_frame.iloc[:train_end].median().fillna(0.0)
feature_scaler.fit(feature_frame.iloc[:train_end].fillna(feature_medians))
prepared = pd.DataFrame(feature_scaler.transform(feature_frame.fillna(feature_medians)), index=df.index, columns=FEATURE_COLUMNS).fillna(0).astype("float32")

target_training = df.iloc[:train_end][TARGET_COLUMNS].replace([np.inf, -np.inf], np.nan)
mor_valid = target_training["VIS_Visibility_MOR"].notna()
if "VIS_SENSOR_ON" in df.columns:
    mor_valid &= df.iloc[:train_end]["VIS_SENSOR_ON"].fillna(0).eq(1)
target_training.loc[~mor_valid, "VIS_Visibility_MOR"] = np.nan
target_medians = target_training.median().fillna(0.0)
target_scaler.fit(target_training.fillna(target_medians))

values = prepared.to_numpy(dtype="float32")
targets = df[TARGET_COLUMNS].replace([np.inf, -np.inf], np.nan).to_numpy(dtype="float32")
mor_measured = df["VIS_Visibility_MOR"].notna().to_numpy(dtype=bool)
mor_sensor_on = df["VIS_SENSOR_ON"].fillna(1).eq(1).to_numpy(dtype=bool) if "VIS_SENSOR_ON" in df.columns else mor_measured

cloud_loss_mask = np.isfinite(targets[:, 0])
mor_loss_mask = mor_measured & mor_sensor_on & np.isfinite(targets[:, 1])
loss_mask = np.column_stack([cloud_loss_mask, mor_loss_mask])
gaps = df["ROW_GAP"].to_numpy(dtype=bool)

window_ends, timestamps = [], []
target_medians_array = target_medians.to_numpy(dtype="float32")
for end in range(LOOKBACK, len(df) - HORIZON + 1):
    lookback_start = end - LOOKBACK
    horizon_stop = end + HORIZON
    if gaps[lookback_start:end + 1].any():
        continue
    if gaps[end:horizon_stop].any():
        continue
    horizon_mask = loss_mask[end:horizon_stop]
    if not horizon_mask.any():
        continue
    window_ends.append(end)
    timestamps.append(df.index[horizon_stop - 1])

window_ends = np.asarray(window_ends, dtype="int64")
timestamps = pd.to_datetime(timestamps)
# Keep full-timeline targets and masks 2D; each horizon is sliced dynamically.
filled_targets = targets.copy()
for target_index in range(len(TARGET_COLUMNS)):
    filled_targets[~loss_mask[:, target_index], target_index] = target_medians_array[target_index]
y_scaled = target_scaler.transform(filled_targets).astype("float32")
mask = loss_mask.astype("float32")

# Maintain at least HORIZON steps of guard space around split boundaries.
train = window_ends < (train_end - HORIZON)
validation = (window_ends >= train_end + HORIZON) & (window_ends < validation_end - HORIZON)
test = window_ends >= validation_end + HORIZON

print("Windows:", len(window_ends), "train/validation/test:", int(train.sum()), int(validation.sum()), int(test.sum()))


c:\Users\iiiaamnooob\miniconda3\envs\weather_pipe\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Windows: 2817552 train/validation/test: 1981061 425533 410478


In [4]:
class MultiOutputLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(len(FEATURE_COLUMNS), 128, num_layers=2, dropout=0.1, batch_first=True)
        self.head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, HORIZON * len(TARGET_COLUMNS)))
    def forward(self, values):
        output, _ = self.lstm(values)
        batch_size = values.size(0)
        forecast = self.head(output[:, -1, :])
        return forecast.reshape(batch_size, HORIZON, len(TARGET_COLUMNS))

class WindowDataset(torch.utils.data.Dataset):
    def __len__(self):
        return len(window_ends)
    def __getitem__(self, index):
        end = window_ends[index]
        x = values[end - LOOKBACK:end]
        y = y_scaled[end:end + HORIZON]
        m = mask[end:end + HORIZON]
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(m)

dataset = WindowDataset()
def loader(split_mask):
    indices = np.flatnonzero(split_mask).tolist()
    return DataLoader(Subset(dataset, indices), batch_size=BATCH_SIZE, shuffle=False)

train_loader = loader(train)
validation_loader = loader(validation)
model = MultiOutputLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction="none")

best_loss = float("inf")
best_state = None
no_improvement = 0
for epoch in range(EPOCHS):
    model.train()
    for inputs, labels, label_mask in train_loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        label_mask = label_mask.to(DEVICE)
        optimizer.zero_grad()
        loss = (criterion(model(inputs), labels) * label_mask).sum() / label_mask.sum().clamp_min(1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    model.eval()
    losses = []
    with torch.no_grad():
        for inputs, labels, label_mask in validation_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            label_mask = label_mask.to(DEVICE)
            loss = (criterion(model(inputs), labels) * label_mask).sum() / label_mask.sum().clamp_min(1)
            losses.append(loss.item())

    validation_loss = float(np.mean(losses)) if losses else float("inf")
    print(f"Epoch {epoch + 1:03d}/{EPOCHS} - val_loss: {validation_loss:.6f}")
    if validation_loss < best_loss:
        best_loss = validation_loss
        no_improvement = 0
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    else:
        no_improvement += 1
        if no_improvement >= PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
torch.save({
    "model_state_dict": model.state_dict(),
    "features": FEATURE_COLUMNS,
    "targets": TARGET_COLUMNS,
    "horizon": HORIZON,
    "lookback": LOOKBACK,
}, ARTIFACT_DIR / "best_guwahati_cloud_lstm_stage4.pt")


Epoch 001/50 - val_loss: 0.401679
Epoch 002/50 - val_loss: 0.364723
Epoch 003/50 - val_loss: 0.325911
Epoch 004/50 - val_loss: 0.334924
Epoch 005/50 - val_loss: 0.308295
Epoch 006/50 - val_loss: 0.329737
Epoch 007/50 - val_loss: 0.304917
Epoch 008/50 - val_loss: 0.291402
Epoch 009/50 - val_loss: 0.302182
Epoch 010/50 - val_loss: 0.306957
Epoch 011/50 - val_loss: 0.295554
Epoch 012/50 - val_loss: 0.286280
Epoch 013/50 - val_loss: 0.305331
Epoch 014/50 - val_loss: 0.301991
Epoch 015/50 - val_loss: 0.307005
Epoch 016/50 - val_loss: 0.288480
Epoch 017/50 - val_loss: 0.317046
Epoch 018/50 - val_loss: 0.290852
Epoch 019/50 - val_loss: 0.266279
Epoch 020/50 - val_loss: 0.295112
Epoch 021/50 - val_loss: 0.338528
Epoch 022/50 - val_loss: 0.328712
Epoch 023/50 - val_loss: 0.282492
Epoch 024/50 - val_loss: 0.291626
Epoch 025/50 - val_loss: 0.276876
Epoch 026/50 - val_loss: 0.285357
Epoch 027/50 - val_loss: 0.270770
Early stopping.


In [5]:
test_loader = loader(test)
model.eval()
prediction_batches = []
label_batches = []
mask_batches = []
with torch.no_grad():
    for inputs, labels, label_mask in test_loader:
        prediction_batches.append(model(inputs.to(DEVICE)).cpu().numpy())
        label_batches.append(labels.numpy())
        mask_batches.append(label_mask.numpy())
predictions_scaled = np.concatenate(prediction_batches, axis=0).astype("float32")
labels_scaled = np.concatenate(label_batches, axis=0).astype("float32")
label_mask_3d = np.concatenate(mask_batches, axis=0).astype("float32")

predictions = target_scaler.inverse_transform(predictions_scaled.reshape(-1, len(TARGET_COLUMNS))).reshape(predictions_scaled.shape)
actual = target_scaler.inverse_transform(labels_scaled.reshape(-1, len(TARGET_COLUMNS))).reshape(labels_scaled.shape)

def masked_metrics(actual_3d, predicted_3d, mask_3d, target_index):
    valid = mask_3d[:, :, target_index].astype(bool)
    actual_target = actual_3d[:, :, target_index][valid]
    predicted_target = predicted_3d[:, :, target_index][valid]
    if len(actual_target) == 0:
        return {"measured_samples": 0, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan}
    return {
        "measured_samples": len(actual_target),
        "MAE": mean_absolute_error(actual_target, predicted_target),
        "RMSE": mean_squared_error(actual_target, predicted_target) ** 0.5,
        "R2": r2_score(actual_target, predicted_target) if len(actual_target) > 1 else np.nan,
    }

lead_times = [("+5 min", 19), ("+15 min", 59), ("+30 min", 119)]
rows = []
for target_index, target in enumerate(TARGET_COLUMNS):
    rows.append({"target": target, "lead_time": "overall", **masked_metrics(actual, predictions, label_mask_3d, target_index)})
    for lead_name, lead_index in lead_times:
        valid = label_mask_3d[:, lead_index, target_index].astype(bool)
        actual_lead = actual[valid, lead_index, target_index]
        predicted_lead = predictions[valid, lead_index, target_index]
        if len(actual_lead) == 0:
            metric = {"measured_samples": 0, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan}
        else:
            metric = {
                "measured_samples": len(actual_lead),
                "MAE": mean_absolute_error(actual_lead, predicted_lead),
                "RMSE": mean_squared_error(actual_lead, predicted_lead) ** 0.5,
                "R2": r2_score(actual_lead, predicted_lead) if len(actual_lead) > 1 else np.nan,
            }
        rows.append({"target": target, "lead_time": lead_name, **metric})

metrics = pd.DataFrame(rows)
metrics.to_csv(ARTIFACT_DIR / "test_metrics.csv", index=False)
display(metrics)


,target,lead_time,measured_samples,MAE,RMSE,R2
0,CLOUD_Vertical_Visibility,overall,49257360,101.781410,885.629966,0.674708
1,CLOUD_Vertical_Visibility,+5 min,410478,88.022209,697.696231,0.794679
2,CLOUD_Vertical_Visibility,+15 min,410478,105.321648,896.477830,0.666462
3,CLOUD_Vertical_Visibility,+30 min,410478,139.486404,1118.779748,0.494496
4,VIS_Visibility_MOR,overall,43586135,1599.430664,2042.446572,0.514686
5,VIS_Visibility_MOR,+5 min,363188,1585.254150,2024.564336,0.523338
6,VIS_Visibility_MOR,+15 min,363228,1598.708984,2042.263573,0.514829
7,VIS_Visibility_MOR,+30 min,363238,1619.575195,2066.009075,0.503119


In [6]:
# Clean 30-minute inference example (120 future 15-second steps)
checkpoint = torch.load(ARTIFACT_DIR / "best_guwahati_cloud_lstm_stage4.pt", map_location=DEVICE, weights_only=False)
model = MultiOutputLSTM().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

if len(df) < LOOKBACK:
    raise ValueError("Not enough rows for inference.")
if gaps[-LOOKBACK:].any():
    raise ValueError("The latest LOOKBACK window crosses a ROW_GAP; no inference produced.")

inference_input = torch.from_numpy(values[-LOOKBACK:].copy()).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    scaled_forecast = model(inference_input).cpu().numpy()
forecast_original = target_scaler.inverse_transform(
    scaled_forecast.reshape(-1, len(TARGET_COLUMNS))
).reshape(HORIZON, len(TARGET_COLUMNS))

future_timestamps = pd.date_range(
    start=df.index[-1] + pd.Timedelta(seconds=15),
    periods=HORIZON,
    freq="15s",
)
forecast_df = pd.DataFrame(forecast_original, index=future_timestamps, columns=TARGET_COLUMNS)
forecast_df.index.name = "Datetime"
display(forecast_df)


,CLOUD_Vertical_Visibility,VIS_Visibility_MOR
Datetime,,
2026-01-31 10:09:00,15003.534180,8288.016602
2026-01-31 10:09:15,14986.401367,8276.616211
2026-01-31 10:09:30,14976.960938,8270.612305
2026-01-31 10:09:45,14983.585938,8262.373047
2026-01-31 10:10:00,14985.093750,8253.897461
...,...,...
2026-01-31 10:37:45,14968.490234,7159.228516
2026-01-31 10:38:00,14964.142578,7148.967285
2026-01-31 10:38:15,14960.258789,7139.797363
